In [6]:
import datetime
import sys, os, logging

from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline 
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

# Project root
PROJECT_ROOT = Path(os.path.abspath("../../.."))
sys.path.insert(0, str(PROJECT_ROOT))

from inference.prefilter_model.utils import (
    list_h5_parts, load_model, load_mc_test_parts,
    load_exp_parts, load_mc_reco_parts, predict_scores,
    plot_event_3d, load_blind_reco,
    EXP_RECO_COL_NAMES, MC_RECO_COL_NAMES, COMMON_RECO_COLS,
)

%load_ext autoreload
%autoreload 2

In [7]:
# ── Paths ──

MODEL_NAME = "da_prefilter_numu_260410_1119_Qclip_softlabels_PlateauLR_lambda0.3_MC2M_dmodel128_zflipBothMCExp_FocalLoss"
CHECKPOINT_PATH = f"experiments/numu/{MODEL_NAME}/best_da_model.pth"

EXP_RECO_H5_PATH = "data_manager/data/h5datasets/exp_reco.h5"
BLIND_ANALYSIS_ROOTS = "data_manager/data/exp_reco_root/reco_blind_files"

# ── Figures output directory ──
TIMESTAMP_STR = datetime.datetime.now().strftime("%y%m%d_%H%M%S")
FIGURES_DIR = PROJECT_ROOT / f"experiments/numu/{MODEL_NAME}/test_reco_figures_{TIMESTAMP_STR}"
#FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, filename: str, rewrite=True) -> None:
    """Save figure to FIGURES_DIR; raise FileExistsError if already present."""
    fig_path = FIGURES_DIR / filename
    if not rewrite and fig_path.exists():
        raise FileExistsError(f"Figure already exists: {fig_path}")
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {fig_path}")

# ── MC test parts to load (from complement of training parts) ──
N_PARTS_PER_PARTICLE = {
    "muatm": 16,
    "nue2": 8,
    "nuatm": 8,
}

reco_parts = list_h5_parts(str(PROJECT_ROOT / EXP_RECO_H5_PATH), 'exp_reco')
BAD_RECO_PARTS = [
    p for p in reco_parts if 'c01' in p
]

# ── Inference settings ──
DEVICE = "cuda:4"
BATCH_SIZE = 1024
MAX_HITS = 500
SEED = 42

# ── reco_prty column mapping (index -> name) (jsut in case) ──
RECO_EVENTS_COLUMNS = {
    0: "fThetaRec",
    1: "fPhiRec",
    2: "fThetaErr",
    3: "fPhiErr",
    4: "fFuncValue",
    5: "fTimeChi2",
    6: "fChargeTerm",
    7: "fLLFit",
    8: "fNHits",
    9: "fNStrings",
    10: "fNOMs",
    11: "fPathLength",
    12: "fTimeXYZRec",
    13: "fCovMatrixStatus",
    14: "fScfMaxTheta",
    15: "fScfMinTheta",
    16: "fScfTheta",
    17: "fScfPhi",
    18: "fPHit",
    19: "fEvCenterZ",
    20: "fZDist",
    21: "fNTriplets",
    22: "fNCalls",
    23: "fClassBDT", # Just placeholders...
    24: "fClassBDTLowE", # Just placeholders...
}
RECO_HITS_COLUMNS = { 
    0: "fTRes" # Unmatchable with other hits data!!!
}

# Load data

In [8]:
# ── Limiters (set to None for full dataset) ──
MAX_PARTS_EXP = None        # limit parts read from exp_reco.h5
MAX_PARTS_MC  = None        # limit parts per particle type from baikal_mc_reco.h5

In [9]:
# ── Load exp_reco ──
df_exp = load_exp_parts(
    h5_path=str(PROJECT_ROOT / EXP_RECO_H5_PATH),
    group_name="exp_reco",
    max_parts=MAX_PARTS_EXP,
    load_reco=True,
)
print(f"exp_reco: {len(df_exp):,} events")
print(f"  reco columns: {[c for c in df_exp.columns if c in EXP_RECO_COL_NAMES]}")
df_exp.head(3)

Exp exp_reco:   0%|          | 0/370 [00:00<?, ?part/s]

2026-04-17 08:05:46,886 INFO Loaded 14,447,866 events from /home/albert/Baikal2025/data_manager/data/h5datasets/exp_reco.h5 (exp_reco)


exp_reco: 14,447,866 events
  reco columns: ['thetaRec', 'phiRec', 'thetaErr', 'phiErr', 'funcValue', 'timeChi2', 'chargeTerm', 'LLFit', 'nHits', 'nStrings', 'nOMs', 'pathLength', 'timeXYZRec', 'covMatrixStatus', 'scfMaxTheta', 'scfMinTheta', 'scfTheta', 'scfPhi', 'pHit', 'evCenterZ', 'zDist', 'nTriplets', 'nCalls', 'classBDT', 'classBDTLowE']


,h5_part_str,local_id,n_hits,features,signal_mask,reco_hit_prty,season,cluster,run,event_id_cc,...,scfMinTheta,scfTheta,scfPhi,pHit,evCenterZ,zDist,nTriplets,nCalls,classBDT,classBDTLowE
0,part_s2020_c01_r0001,0,70,"[[1.0081818, -14357.419, -59.7987, -11.325296,...","[False, False, False, False, False, False, Fal...","[[0], [0], [0], [0], [0], [0], [0], [0], [0], ...",2020,1,1,137,...,2.722713,2.827433,1.570796,5.976916e-04,312.461060,224.639542,1.0,137.0,-2.0,-2.0
1,part_s2020_c01_r0001,1,81,"[[1.5283558, -14132.793, -60.522926, -11.87183...","[False, False, False, False, False, False, Fal...","[[0], [0], [0], [0], [0], [0], [0], [0], [0], ...",2020,1,1,234,...,2.827433,3.036872,1.047198,8.037232e-09,164.412994,405.534088,0.0,327.0,-2.0,-2.0
2,part_s2020_c01_r0001,2,77,"[[0.97016454, -14071.478, -59.782684, -11.2957...","[False, False, False, False, False, False, Fal...","[[0], [0], [0], [0], [0], [0], [0], [0], [-4],...",2020,1,1,298,...,1.884956,1.989675,3.769909,5.781715e-05,554.948547,146.446915,2.0,186.0,-2.0,-2.0


In [10]:
MC_RECO_H5_PATH = "data_manager/data/h5datasets/baikal_mc_reco.h5"
MC_PARTICLE_TYPES = ["muatm", "nuatm_conv"]  # adjust if nue2 is available

# ── Load mc_reco ──
# NOTE: baikal_mc_reco.h5 must be regenerated with the updated root2h5_config_mc_reco.yaml
# (25 common reco columns + 6 MC-specific vector columns = 31 total).
# If the file still has the old 21-column format, reco columns will be named reco_0..reco_20.
df_mc = load_mc_reco_parts(
    h5_path=str(PROJECT_ROOT / MC_RECO_H5_PATH),
    particle_types=MC_PARTICLE_TYPES,
    max_parts_per_particle=MAX_PARTS_MC,
)
print(f"mc_reco: {len(df_mc):,} events")
print(f"  particle counts:\n{df_mc['particle_type'].value_counts().to_string()}")
print(f"  reco columns: {[c for c in df_mc.columns if c in MC_RECO_COL_NAMES]}")
df_mc.head(3)

2026-04-17 08:05:59,051 INFO Loading 11064 parts for 'muatm'


MC-reco muatm:   0%|          | 0/11064 [00:00<?, ?part/s]

2026-04-17 08:07:25,296 INFO Loading 11833 parts for 'nuatm_conv'


MC-reco nuatm_conv:   0%|          | 0/11833 [00:00<?, ?part/s]

2026-04-17 08:08:07,241 INFO MC-reco: 3,228,595 events from /home/albert/Baikal2025/data_manager/data/h5datasets/baikal_mc_reco.h5 (muatm, nuatm_conv)


mc_reco: 3,228,595 events
  particle counts:
particle_type
muatm         2597359
nuatm_conv     631236
  reco columns: ['thetaRec', 'phiRec', 'thetaErr', 'phiErr', 'funcValue', 'timeChi2', 'chargeTerm', 'LLFit', 'nHits', 'nStrings', 'nOMs', 'pathLength', 'timeXYZRec', 'covMatrixStatus', 'scfMaxTheta', 'scfMinTheta', 'scfTheta', 'scfPhi', 'pHit', 'evCenterZ', 'zDist', 'nTriplets', 'nCalls', 'classBDT', 'classBDTLowE', 'xyzRec_x', 'xyzRec_y', 'xyzRec_z', 'dirRec_x', 'dirRec_y', 'dirRec_z']


,particle_type,h5_part_str,local_id,n_hits,features,signal_mask,theta_mc,phi_mc,energy_mc,nucleon_n,...,nTriplets,nCalls,classBDT,classBDTLowE,xyzRec_x,xyzRec_y,xyzRec_z,dirRec_x,dirRec_y,dirRec_z
0,muatm,part_2020_cl1_run10000_scl_nu_MC_s19-21,0,82,"[[1.2319751, -2369.2236, 17.954746, -55.696243...","[False, False, False, False, False, False, Fal...",134.918213,275.348419,489.288025,402.0,...,2.0,71.0,-2.0,-2.0,218.438797,93.587463,-38.113953,0.110401,-0.584927,-0.803537
1,muatm,part_2020_cl1_run10000_scl_nu_MC_s19-21,1,73,"[[0.78172624, -2505.7773, -42.039257, 38.46075...","[False, False, False, False, False, False, Fal...",129.888229,30.562561,2188.936035,2010.0,...,0.0,124.0,-2.0,-2.0,229.389465,112.331108,185.835190,0.170086,0.736250,-0.654987
2,muatm,part_2020_cl1_run10000_scl_nu_MC_s19-21,2,66,"[[9.61653, -2528.6812, -42.039257, 38.46075, 1...","[False, False, False, False, False, False, Fal...",115.446831,333.445129,681.872314,14.0,...,4.0,117.0,-2.0,-2.0,239.782288,199.493622,3.547806,0.576949,-0.685570,-0.443986


In [11]:
# ── Verify matchable reco columns ──
exp_reco_cols = [c for c in df_exp.columns if c in set(EXP_RECO_COL_NAMES)]
mc_reco_cols  = [c for c in df_mc.columns  if c in set(MC_RECO_COL_NAMES)]
common = [c for c in exp_reco_cols if c in set(mc_reco_cols)]

print(f"exp_reco  reco columns ({len(exp_reco_cols)}): {exp_reco_cols}")
print(f"mc_reco   reco columns ({len(mc_reco_cols)}):  {mc_reco_cols}")
print(f"\nCommon matchable columns ({len(common)}): {common}")

mc_only = [c for c in mc_reco_cols if c not in set(exp_reco_cols)]
exp_only = [c for c in exp_reco_cols if c not in set(mc_reco_cols)]
if mc_only:
    print(f"MC-only reco columns: {mc_only}")
if exp_only:
    print(f"exp-only reco columns: {exp_only}")

exp_reco  reco columns (25): ['thetaRec', 'phiRec', 'thetaErr', 'phiErr', 'funcValue', 'timeChi2', 'chargeTerm', 'LLFit', 'nHits', 'nStrings', 'nOMs', 'pathLength', 'timeXYZRec', 'covMatrixStatus', 'scfMaxTheta', 'scfMinTheta', 'scfTheta', 'scfPhi', 'pHit', 'evCenterZ', 'zDist', 'nTriplets', 'nCalls', 'classBDT', 'classBDTLowE']
mc_reco   reco columns (31):  ['thetaRec', 'phiRec', 'thetaErr', 'phiErr', 'funcValue', 'timeChi2', 'chargeTerm', 'LLFit', 'nHits', 'nStrings', 'nOMs', 'pathLength', 'timeXYZRec', 'covMatrixStatus', 'scfMaxTheta', 'scfMinTheta', 'scfTheta', 'scfPhi', 'pHit', 'evCenterZ', 'zDist', 'nTriplets', 'nCalls', 'classBDT', 'classBDTLowE', 'xyzRec_x', 'xyzRec_y', 'xyzRec_z', 'dirRec_x', 'dirRec_y', 'dirRec_z']

Common matchable columns (25): ['thetaRec', 'phiRec', 'thetaErr', 'phiErr', 'funcValue', 'timeChi2', 'chargeTerm', 'LLFit', 'nHits', 'nStrings', 'nOMs', 'pathLength', 'timeXYZRec', 'covMatrixStatus', 'scfMaxTheta', 'scfMinTheta', 'scfTheta', 'scfPhi', 'pHit', 'evC

# Load model

In [12]:
model, norm_config, train_config = load_model(
    str(PROJECT_ROOT / CHECKPOINT_PATH), device=DEVICE
)
print(f"Normalization: means={norm_config['means']}, stds={norm_config['stds']}")
print(f"Model amp_clip: {model.amp_clip:.4f} (normalized) "
      f"= Q≈{model.amp_clip * norm_config['stds'][0] + norm_config['means'][0]:.1f} PE")


2026-04-17 08:08:09,404 INFO Created AttentionFeatureExtractor: 5→128, 4 layers, 4 heads, pooling=cls
2026-04-17 08:08:09,405 INFO Created BinaryClassifier: 128→[32, 16]→1, dropout=0.0, batch_norm=False
2026-04-17 08:08:09,406 INFO Created StandardNeutrinoModel with 404161 parameters
2026-04-17 08:08:09,408 INFO amp_clip: Q=100 PE → normalized=1.6333 (mean=2.0, std=60.0)
2026-04-17 08:08:09,411 INFO Loaded model from epoch 55 (best_metric=0.9982)


Normalization: means=[2.0, 0.0, 0.0, 0.0, 0.0], stds=[60.0, 1370.0, 40.0, 40.0, 150.0]
Model amp_clip: 1.6333 (normalized) = Q≈100.0 PE


# Predict scores

In [ ]:
print("Prediction...")
df_exp["score"] = predict_scores(
    model, df_exp["features"].tolist(), norm_config,
    batch_size=BATCH_SIZE, max_hits=MAX_HITS, device=DEVICE,
)
print(f"Exp reco scores: mean={df_exp['score'].mean():.4f}, median={df_exp['score'].median():.4f}")

Prediction...


Predicting:   0%|          | 0/14110 [00:00<?, ?batch/s]

/home/albert/miniconda3/envs/baikal25/lib/python3.10/site-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


In [1]:
print("Prediction...")
df_mc["score"] = predict_scores(
    model, df_mc["features"].tolist(), norm_config,
    batch_size=BATCH_SIZE, max_hits=MAX_HITS, device=DEVICE,
)
print(f"MC reco scores: mean={df_mc['score'].mean():.4f}, median={df_mc['score'].median():.4f}")

Prediction...


NameError: name 'predict_scores' is not defined

# Save results

In [ ]:
import pyarrow as pa
import pyarrow.feather as feather

# ── Helpers ──────────────────────────────────────────────────────────────────
def _series_to_arrow(series: pd.Series) -> pa.Array:
    sample = series.iloc[0]
    pa_dtype = pa.from_numpy_dtype(sample.dtype)
    lengths = np.fromiter((len(x) for x in series), dtype=np.int32, count=len(series))
    offsets = np.empty(len(series) + 1, dtype=np.int32)
    offsets[0] = 0
    np.cumsum(lengths, out=offsets[1:])
    flat = np.concatenate(series.values)
    values = (
        pa.array(flat, type=pa_dtype) if sample.ndim == 1
        else pa.FixedSizeListArray.from_arrays(pa.array(flat.ravel(), type=pa_dtype), sample.shape[1])
    )
    return pa.ListArray.from_arrays(offsets, values)

def _df_to_arrow(df: pd.DataFrame) -> pa.Table:
    col_arrays = {}
    for col in df.columns:
        sample = df[col].iloc[0]
        col_arrays[col] = (
            _series_to_arrow(df[col]) if isinstance(sample, np.ndarray)
            else pa.array(df[col].to_numpy())
        )
    return pa.table(col_arrays)

# ── Save ──────────────────────────────────────────────────────────────────────
SAVE_DIR = Path(f"./{MODEL_NAME}")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

ARRAY_COLS = {"features", "signal_mask", "reco_hit_prty"}
JOIN_KEYS  = ["h5_part_str", "local_id"]  # match scalars ↔ arrays

for name, df in [("exp", df_exp), ("mc", df_mc)]:
    scalar_cols = [c for c in df.columns if c not in ARRAY_COLS]
    array_cols  = [c for c in df.columns if c in ARRAY_COLS]

    # Scalars → parquet (supports predicate pushdown)
    df[scalar_cols].to_parquet(SAVE_DIR / f"{name}_scalars.parquet", index=False)

    # Arrays → feather (fast binary, join keys included)
    feather.write_feather(
        _df_to_arrow(df[JOIN_KEYS + array_cols]),
        SAVE_DIR / f"{name}_arrays.arrow",
    )

print(f"Saved to {SAVE_DIR}/")
print(f"  exp_scalars.parquet  +  exp_arrays.arrow")
print(f"  mc_scalars.parquet   +  mc_arrays.arrow")
print(f"Join key: {JOIN_KEYS}")


# Tests

In [ ]:
df_exp_scalars = pd.read_parquet(SAVE_DIR / "exp_scalars.parquet")
df_exp_arrays  = feather.read_table(SAVE_DIR / "exp_arrays.arrow").to_pandas()
df_exp = df_exp_scalars.merge(df_exp_arrays, on=JOIN_KEYS)